# 👁️ Gaze Tracker & Attention Monitor (V5 - SENSÍVEL)

Monitoramento de atenção para o seu projeto com correção para o erro de acentos no Windows.

### Instruções:
1. Abaixo, carregamos o arquivo `models/face_landmarker.task` como um buffer de memória.
2. Rode a célula de monitoramento.

In [ ]:
import cv2
import numpy as np
import pygame
import time
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

print("Carregando modelo para a memória...")
model_path = 'models/face_landmarker.task'
with open(model_path, 'rb') as f:
    model_data = f.read()

# Configuração para versão 0.10.13
base_options = python.BaseOptions(model_asset_buffer=model_data)
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.VIDEO,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

detector = vision.FaceLandmarker.create_from_options(options)
pygame.mixer.init()
print("✅ Pygame Mixer pronto!")
print("✅ Face Landmarker pronto!")


In [ ]:
cap = cv2.VideoCapture(0)

# 1. Carregar Playlist de Sons (sound1 normal, outros amplificados)
sound_files = [
    ("Alarme (Normal)", "sounds/sound1.wav"),
    ("Chimes (500%)", "sounds/sound2.wav"),
    ("Chord (500%)", "sounds/sound3.wav"),
    ("Ding (500%)", "sounds/sound4.wav"),
    ("Sirene (500%)", "sounds/sound5.wav")
]
sounds_objects = [pygame.mixer.Sound(f[1]) for f in sound_files]

ui_config = {
    "volume": 100,
    "sensibilidade": 35,
    "atraso": 5.0,
    "dragging": None,
    "show_config": False,
    "sound_idx": 0
}

def mouse_callback(event, x, y, flags, param):
    global ui_config
    if event == cv2.EVENT_LBUTTONDOWN:
        if (w-70) <= x <= (w) and 0 <= y <= 60:
            ui_config["show_config"] = not ui_config["show_config"]
            return
            
        if ui_config["show_config"]:
            # Menu de Sons (Suporte para 5 sons)
            for i in range(len(sounds_objects)):
                btn_y_start = 210 + (i * 32) # Espaçamento menor (32px) para caber 5
                if (w-230) <= x <= (w-10) and btn_y_start <= y <= btn_y_start + 28:
                    pygame.mixer.stop()
                    ui_config["sound_idx"] = i
                    sounds_objects[i].set_volume(ui_config["volume"]/100.0)
                    sounds_objects[i].play()
                    return

    if ui_config["show_config"]:
        if event == cv2.EVENT_LBUTTONDOWN or (event == cv2.EVENT_MOUSEMOVE and flags == cv2.EVENT_FLAG_LBUTTON):
            if 85 <= y <= 115 and (w-220) <= x <= (w-20):
                ui_config["volume"] = int(((x - (w-220)) / 200) * 100)
            elif 125 <= y <= 155 and (w-220) <= x <= (w-20):
                ui_config["sensibilidade"] = int(((x - (w-220)) / 200) * 50)
            elif 165 <= y <= 195 and (w-220) <= x <= (w-20):
                ui_config["atraso"] = round(((x - (w-220)) / 200) * 10.0, 1)

cv2.namedWindow('Attention Monitor v5')
cv2.setMouseCallback('Attention Monitor v5', mouse_callback)

last_look = time.time()
is_away = False

while cap.isOpened():
    v, s, d = ui_config["volume"]/100.0, ui_config["sensibilidade"]/100.0, ui_config["atraso"]
    LIMIT_LOW, LIMIT_HIGH, AWAY_TIME = 0.5 - s, 0.5 + s, d
    
    current_label, _ = sound_files[ui_config["sound_idx"]]
    current_sound = sounds_objects[ui_config["sound_idx"]]
    current_sound.set_volume(v)

    success, frame = cap.read()
    if not success: break
    frame = cv2.flip(frame, 1)
    h, w, _ = frame.shape
    overlay = frame.copy()
    
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    timestamp_ms = int(time.time() * 1000)
    result = detector.detect_for_video(mp_image, timestamp_ms)
    away_now = False; status, color = "FOCADO", (0, 255, 100)

    if result.face_landmarks:
        landmarks = result.face_landmarks[0]
        nose, left, right = landmarks[1], landmarks[33], landmarks[263]
        prop = (nose.x - left.x) / (right.x - left.x)
        away_now = prop < LIMIT_LOW or prop > LIMIT_HIGH
        if away_now:
            time_away = time.time() - last_look if is_away else 0
            if time_away > AWAY_TIME:
                if not pygame.mixer.get_busy(): current_sound.play()
                status, color = f"DISTRAIDO! ({time_away:.1f}s)", (50, 50, 255)
            else: status, color = "ATENCAO...", (0, 180, 255)
        for idx in [468, 473]:
            lm = landmarks[idx]
            cv2.circle(frame, (int(lm.x*w), int(lm.y*h)), 6, (255, 255, 255), -1)
            cv2.circle(frame, (int(lm.x*w), int(lm.y*h)), 4, (255, 255, 0), -1)
    else:
        away_now = True
        time_away = time.time() - last_look if is_away else 0
        if time_away > AWAY_TIME:
            if not pygame.mixer.get_busy(): current_sound.play()
            status, color = "ROSTO PERDIDO!", (50, 50, 255)
        else: status, color = "PROCURANDO...", (0, 180, 255)

    if away_now:
        if not is_away: is_away = True; last_look = time.time()
    else:
        if is_away: is_away = False; current_sound.fadeout(500)

    # --- HUD ---
    cv2.rectangle(overlay, (0, 0), (w, 60), (20, 20, 20), -1)
    btn_x, btn_y = w-45, 30
    btn_color = (180, 180, 180) if not ui_config["show_config"] else color
    cv2.circle(overlay, (btn_x, btn_y), 15, (60, 60, 60), -1)
    cv2.circle(overlay, (btn_x, btn_y), 8, btn_color, 2)
    for i in range(8):
        import math
        angle = i * (45) * (math.pi/180)
        p1 = (int(btn_x + 8 * math.cos(angle)), int(btn_y + 8 * math.sin(angle)))
        p2 = (int(btn_x + 14 * math.cos(angle)), int(btn_y + 14 * math.sin(angle)))
        cv2.line(overlay, p1, p2, btn_color, 2)
    
    if is_away:
        prog = min((time.time() - last_look) / AWAY_TIME, 1.0)
        cv2.rectangle(overlay, (0, 55), (int(prog * w), 60), color, -1)
    
    cv2.addWeighted(overlay, 0.7, frame, 0.3, 0, frame)
    cv2.putText(frame, status, (20, 40), cv2.FONT_HERSHEY_DUPLEX, 1.0, color, 2)
    if result.face_landmarks: cv2.putText(frame, f"Gaze: {prop:.2f}", (w-200, 40), 0, 0.5, (200, 200, 200), 1)

    if ui_config["show_config"]:
        # Painel Estendido (380px para caber 5 sons)
        cv2.rectangle(frame, (w-240, 60), (w, 380), (35, 35, 35), -1)
        
        def draw_slider(target_frame, label, val, y, max_val, unit=""):
            x_start, x_end = w-220, w-20
            cv2.line(target_frame, (x_start, y+10), (x_end, y+10), (100, 100, 100), 2)
            knob_pos = x_start + int((val / max_val) * 200)
            cv2.circle(target_frame, (knob_pos, y+10), 8, (255, 255, 255), -1)
            cv2.putText(target_frame, f"{label}: {val}{unit}", (x_start, y-2), 0, 0.4, (255, 255, 255), 1)

        draw_slider(frame, "Volume", ui_config["volume"], 90, 100, "%")
        draw_slider(frame, "Sensibilidade", ui_config["sensibilidade"], 130, 50)
        draw_slider(frame, "Atraso", ui_config["atraso"], 170, 10.0, "s")
        
        cv2.putText(frame, "ESCOLHA O ALERTA:", (w-225, 205), 0, 0.4, (150, 150, 150), 1)
        for i, (label, _) in enumerate(sound_files):
            btn_y = 210 + (i * 32)
            is_selected = i == ui_config["sound_idx"]
            bg_color, txt_color = ((80, 80, 80), (255, 255, 255)) if not is_selected else (color, (0, 0, 0))
            cv2.rectangle(frame, (w-230, btn_y), (w-10, btn_y + 28), bg_color, -1)
            cv2.putText(frame, label, (w-220, btn_y + 20), 0, 0.4, txt_color, 1)

    cv2.imshow('Attention Monitor v5', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
detector.close()
